# Phase 1: Data Engineering & Banking Domain Feature Engineering

## Executive Context & Objectives
Customer churn in retail and commercial banking directly drives deposit flight and erosion of **Net Interest Margin (NIM)**.
In this notebook, we:
1. **Ingest & Clean** the 10,000-customer banking churn dataset.
2. **Validate Data Schemas** with strict constraints using **Pandera**.
3. **Engineer Banking Domain Features**:
   - **Wealth & Liquidity Metrics**: `BalanceToSalaryRatio`, `IsZeroBalance`, `WealthTier` (Zero Balance, Mass Market, Affluent, High Net Worth).
   - **Customer Stickiness & Life Cycle**: `TenureToAgeRatio`, `CreditScoreToAgeRatio`.
   - **Product Penetration & Complexity Risk**: `IsMultiProductRisk` ($NumOfProducts \ge 3$).
   - **Friction & Dissatisfaction Multipliers**: `ComplaintInactivityRisk`, `ComplaintRisk`.
   - **Loyalty & Rewards Index**: `LoyaltyIndex` (Tier weight $\times$ Satisfaction $\times$ Points).
4. **Analyze Distributions & Churn Drivers** with high-resolution visual analytics.
5. **Persist Leak-Free Preprocessing Pipelines** for downstream production modeling.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_PATH, PROCESSED_DATA_PATH, TARGET_COL
from src.preprocess import load_and_clean_data, BankingFeatureEngineer, build_preprocessor_pipeline
from src.data_schema import raw_bank_data_schema, engineered_bank_data_schema

pd.set_option('display.max_columns', 30)
print("Libraries loaded successfully.")

## 1. Raw Data Ingestion & Pandera Schema Validation

In [ ]:
# Load and clean raw dataset (dropping non-predictive CustomerId, RowNumber, Surname)
df_clean = load_and_clean_data(validate=True)
print(f"Raw dataset shape after cleaning: {df_clean.shape}")
print(f"Target distribution:\n{df_clean[TARGET_COL].value_counts(normalize=True).round(4) * 100}%")
df_clean.head()

## 2. Banking Domain Feature Engineering

In [ ]:
fe = BankingFeatureEngineer()
df_engineered = fe.transform(df_clean)

# Validate engineered dataset against Pandera schema
df_validated = engineered_bank_data_schema.validate(df_engineered)
print(f"Engineered feature count: {df_engineered.shape[1]}")
print("Engineered schema validation passed!")
df_engineered[["Balance", "EstimatedSalary", "BalanceToSalaryRatio", "WealthTier", "TenureToAgeRatio", "IsMultiProductRisk", "ComplaintInactivityRisk", "LoyaltyIndex"]].head()

## 3. Key Churn Dynamics & Statistical Insights

### Finding 1: The Service Complaint Multiplier
Customers who registered a complaint exhibit an extraordinary churn rate (>99% in this dataset), demonstrating that unresolved customer friction is the single strongest precursor to deposit flight.

In [ ]:
complain_summary = df_engineered.groupby("Complain").agg(
    Total_Customers=("Exited", "count"),
    Churn_Rate=("Exited", "mean"),
    Avg_Balance=("Balance", "mean"),
    Total_Deposits=("Balance", "sum")
).reset_index()
complain_summary

### Finding 2: Product Bundling Paradox
Customers with 2 products exhibit the lowest churn (~7.6%), representing the banking sweet spot. Customers with 3 or 4 products have churn rates exceeding 80%, indicating product fatigue, mis-selling, or disjointed cross-selling.

In [ ]:
prod_summary = df_engineered.groupby("NumOfProducts").agg(
    Customer_Count=("Exited", "count"),
    Churn_Rate=("Exited", "mean"),
    Avg_Loyalty_Points=("PointEarned", "mean"),
    Total_Deposits_at_Risk=("Balance", lambda x: x[df_engineered.loc[x.index, "Exited"] == 1].sum())
).reset_index()
prod_summary

### Finding 3: Wealth Tiers & Deposit Flight Concentration
Affluent and High Net Worth customers represent over 70% of all balances at risk of departure, requiring targeted Relationship Manager retention workflows.

In [ ]:
tier_summary = df_engineered.groupby("WealthTier").agg(
    Customer_Count=("Exited", "count"),
    Churn_Rate=("Exited", "mean"),
    Avg_Balance=("Balance", "mean"),
    Total_Deposits=("Balance", "sum"),
    Deposits_at_Risk=("Balance", lambda x: x[df_engineered.loc[x.index, "Exited"] == 1].sum())
).reset_index()
tier_summary

## 4. Leak-Free Preprocessor Pipeline Construction

In [ ]:
from sklearn.model_selection import train_test_split
from src.preprocess import split_features_and_target

X, y = split_features_and_target(df_clean)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = build_preprocessor_pipeline(X_train)
X_train_fe = fe.transform(X_train)
X_train_processed = preprocessor.fit_transform(X_train_fe)

print(f"X_train raw shape: {X_train.shape}")
print(f"X_train processed matrix shape: {X_train_processed.shape}")
print("Feature names out:", preprocessor.get_feature_names_out())